# Data Preparation: Preprocessing and Feature Engineering

## Data Preprocessing

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_PATH = Path("../data/AirQualityUCI.csv")

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=",",
    na_values=-200
)

empty_columns = [
    column
    for column in ["Unnamed: 15", "Unnamed: 16"]
    if column in df.columns
]

df = (
    df
    .drop(columns=empty_columns)
    .dropna(how="all")
    .copy()
)

df["DateTime"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Time"].astype(str).str.replace(".", ":", regex=False),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

df = (
    df
    .drop(columns=["Date", "Time"])
    .sort_values("DateTime")
    .reset_index(drop=True)
)

In [3]:
target_columns = [
    "CO(GT)",
    "C6H6(GT)",
    "NOx(GT)",
    "NO2(GT)",
]

sensor_features = [
    "PT08.S1(CO)",
    "PT08.S2(NMHC)",
    "PT08.S3(NOx)",
    "PT08.S4(NO2)",
    "PT08.S5(O3)",
]

environment_features = [
    "T",
    "RH",
    "AH",
]

ground_truth_columns = [
    "CO(GT)",
    "NMHC(GT)",
    "C6H6(GT)",
    "NOx(GT)",
    "NO2(GT)",
]

In [4]:
df.shape

(9357, 14)

In [5]:
df.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,DateTime
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,2004-03-10 18:00:00
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,2004-03-10 19:00:00
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,2004-03-10 20:00:00
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,2004-03-10 21:00:00
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,2004-03-10 22:00:00


In [6]:
df.dtypes

CO(GT)                  float64
PT08.S1(CO)             float64
NMHC(GT)                float64
C6H6(GT)                float64
PT08.S2(NMHC)           float64
NOx(GT)                 float64
PT08.S3(NOx)            float64
NO2(GT)                 float64
PT08.S4(NO2)            float64
PT08.S5(O3)             float64
T                       float64
RH                      float64
AH                      float64
DateTime         datetime64[us]
dtype: object

In [7]:
print("Invalid DateTime:", df["DateTime"].isna().sum())
print("Duplicate DateTime:", df["DateTime"].duplicated().sum())

Invalid DateTime: 0
Duplicate DateTime: 0


In [8]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Start:", df["DateTime"].min())
print("End:", df["DateTime"].max())
print("Sorted:", df["DateTime"].is_monotonic_increasing)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["DateTime"].duplicated().sum())

time_gap = df["DateTime"].diff().value_counts().head()
time_gap

Rows: 9357
Columns: 14
Start: 2004-03-10 18:00:00
End: 2005-04-04 14:00:00
Sorted: True
Duplicate rows: 0
Duplicate timestamps: 0


DateTime
0 days 01:00:00    9356
Name: count, dtype: int64

## Feature Engineering

In [9]:
df["hour"] = df["DateTime"].dt.hour
df["day_of_week"] = df["DateTime"].dt.dayofweek
df["month"] = df["DateTime"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [10]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

In [ ]:
df["elapsed_hours"] = (df["DateTime"] - df["DateTime"].min()).dt.total_seconds() / 3600
df["elapsed_days"] = df["elapsed_hours"] / 24

In [12]:
df["T_RH"] = df["T"] * df["RH"]
df["T_AH"] = df["T"] * df["AH"]
df["S1_S5_ratio"] = df["PT08.S1(CO)"] / df["PT08.S5(O3)"].replace(0, np.nan)
df["S3_S5_diff"] = df["PT08.S3(NOx)"] - df["PT08.S5(O3)"]
df["S1_S3_diff"] = df["PT08.S1(CO)"] - df["PT08.S3(NOx)"]

In [13]:
time_feature_columns = ["hour", "day_of_week", "month", "is_weekend", "hour_sin", "hour_cos", "month_sin", "month_cos"]
drift_feature_columns = ["elapsed_hours", "elapsed_days"]
interaction_feature_columns = ["T_RH", "T_AH", "S1_S5_ratio", "S3_S5_diff", "S1_S3_diff"]

df[["DateTime"] + time_feature_columns + drift_feature_columns + interaction_feature_columns].head()

,DateTime,hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos,elapsed_hours,elapsed_days,T_RH,T_AH,S1_S5_ratio,S3_S5_diff,S1_S3_diff
0,2004-03-10 18:00:00,18,2,3,0,-1.000000,-1.836970e-16,0.866025,0.5,0.0,0.000000,665.04,10.30608,1.072555,-212.0,304.0
1,2004-03-10 19:00:00,19,2,3,0,-0.965926,2.588190e-01,0.866025,0.5,1.0,0.041667,634.41,9.64915,1.329218,202.0,118.0
2,2004-03-10 20:00:00,20,2,3,0,-0.866025,5.000000e-01,0.866025,0.5,2.0,0.083333,642.60,8.92738,1.305400,66.0,262.0
3,2004-03-10 21:00:00,21,2,3,0,-0.707107,7.071068e-01,0.866025,0.5,3.0,0.125000,660.00,8.65370,1.143807,-111.0,284.0
4,2004-03-10 22:00:00,22,2,3,0,-0.500000,8.660254e-01,0.866025,0.5,4.0,0.166667,667.52,8.83456,1.145946,95.0,67.0


In [14]:
feature_columns = (
    sensor_features
    + environment_features
    + ["hour_sin", "hour_cos", "month_sin", "month_cos", "day_of_week", "is_weekend"]
    + drift_feature_columns
    + interaction_feature_columns
)
feature_columns

['PT08.S1(CO)',
 'PT08.S2(NMHC)',
 'PT08.S3(NOx)',
 'PT08.S4(NO2)',
 'PT08.S5(O3)',
 'T',
 'RH',
 'AH',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'day_of_week',
 'is_weekend',
 'elapsed_hours',
 'elapsed_days',
 'T_RH',
 'T_AH',
 'S1_S5_ratio',
 'S3_S5_diff',
 'S1_S3_diff']

## Time Based Split
The data is divided based on chronological order:
- train: the earliest period.
- validation: the period following the training set.
- test: the final period.

In [15]:
train_ratio = 0.70
validation_ratio = 0.15
test_ratio = 0.15

In [16]:
n_rows = len(df)

train_end = int(n_rows * train_ratio)
validation_end = int(n_rows * (train_ratio + validation_ratio))

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

In [17]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "start": [train_df["DateTime"].min(), validation_df["DateTime"].min(), test_df["DateTime"].min()],
    "end": [train_df["DateTime"].max(), validation_df["DateTime"].max(), test_df["DateTime"].max()],
})

split_summary

,split,rows,start,end
0,train,6549,2004-03-10 18:00:00,2004-12-08 14:00:00
1,validation,1404,2004-12-08 15:00:00,2005-02-05 02:00:00
2,test,1404,2005-02-05 03:00:00,2005-04-04 14:00:00


## Export Split Datasets

In [18]:
export_columns = (
    ["DateTime"]
    + feature_columns
    + target_columns
)

export_columns

['DateTime',
 'PT08.S1(CO)',
 'PT08.S2(NMHC)',
 'PT08.S3(NOx)',
 'PT08.S4(NO2)',
 'PT08.S5(O3)',
 'T',
 'RH',
 'AH',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'day_of_week',
 'is_weekend',
 'elapsed_hours',
 'elapsed_days',
 'T_RH',
 'T_AH',
 'S1_S5_ratio',
 'S3_S5_diff',
 'S1_S3_diff',
 'CO(GT)',
 'C6H6(GT)',
 'NOx(GT)',
 'NO2(GT)']

In [19]:
OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

train_df[export_columns].to_csv(OUTPUT_PATH / "train.csv", index=False)
validation_df[export_columns].to_csv(OUTPUT_PATH / "validation.csv", index=False)
test_df[export_columns].to_csv(OUTPUT_PATH / "test.csv", index=False)